# 06 — Spectre and Meltdown: Microarchitectural Attacks

## What This Is
Spectre (Kocher et al., 2018) and Meltdown (Lipp et al., 2018) are microarchitectural attacks that use speculative execution and cache timing to leak memory across trust boundaries. Meltdown leaks kernel memory to user space; Spectre tricks victim code into leaking its own secrets.

## Why It Matters
Spectre/Meltdown affected virtually every CPU shipping in 2018. Patches required OS updates (KPTI), microcode updates, JIT compiler hardening, and browser sandboxing changes. Performance degraded 5-30% on I/O-intensive workloads. New Spectre variants continue to emerge.

## Where the Community Stands
Spectre-v1 (bounds check bypass), v2 (branch target injection), v3 (Meltdown), v4 (speculative store bypass) are the original variants. Subsequent variants (MDS: RIDL, Fallout, ZombieLoad) attack CPU internal buffers. Hardware mitigations are now standard in new CPUs (eIBRS, IPRED_DIS).

## Conceptual Attack Flow
1. Attacker primes the cache with attacker-controlled data
2. Victim code speculatively accesses out-of-bounds memory (secret)
3. Secret value indexes into a probe array (encoding it in cache state)
4. Attacker measures probe array access times (FLUSH+RELOAD)
5. Fast access reveals which cache line was speculatively touched = secret byte

In [ ]:
import random
import time
from typing import List, Optional

# Conceptual Spectre-v1 simulation
# Real implementation requires specific CPU instructions (clflush, rdtscp)
# This demonstrates the attack LOGIC — not a working exploit

# === Victim's memory layout ===
ARRAY1_SIZE = 16
ARRAY1 = list(range(ARRAY1_SIZE))   # victim's bounds-checked array
SECRET = b'MySecretKey12345'          # victim's secret (adjacent in memory)
ARRAY2 = [0] * 256                   # probe array (256 pages = 1MB)
PAGE_SIZE = 4096

# === Attacker's simulated cache oracle ===
class SpectreCacheModel:
    def __init__(self):
        self.cached: set = set()

    def flush_array2(self) -> None:
        self.cached.discard('array2')

    def speculative_access(self, idx: int, secret_byte: int) -> None:
        """Simulate victim's speculative code path.
        In real Spectre: CPU speculatively runs victim_fn(malicious_x)
        even though bounds check fails, because branch predictor was primed.
        """
        # Speculative: access array2[secret_byte * PAGE_SIZE]
        # This encodes secret_byte into cache state
        self.cached.add(f'array2_{secret_byte}')

    def measure_reload(self) -> Optional[int]:
        """FLUSH+RELOAD: which array2 slot was speculatively loaded?"""
        for i in range(256):
            if f'array2_{i}' in self.cached:
                return i
        return None

# Simulate Spectre byte leak
cache_model = SpectreCacheModel()

print('Spectre-v1 Conceptual Demonstration')
print('=' * 45)
leaked = []

for secret_byte_idx in range(len(SECRET)):
    secret_byte = SECRET[secret_byte_idx]
    cache_model.flush_array2()

    # Attacker: mistrain branch predictor (not simulated — requires real CPU)
    # Attacker: call victim with out-of-bounds x pointing to secret
    cache_model.speculative_access(ARRAY1_SIZE + secret_byte_idx, secret_byte)

    # Attacker: measure cache state
    leaked_byte = cache_model.measure_reload()
    leaked.append(leaked_byte)

leaked_str = bytes(b for b in leaked if b is not None).decode('ascii', 'ignore')
print(f'  Target secret: {SECRET.decode()}')
print(f'  Leaked:        {leaked_str}')
print(f'  Match:         {leaked_str == SECRET.decode()}')

## Meltdown vs Spectre: Key Differences

Meltdown exploits an out-of-order execution bug specific to Intel (and some ARM) CPUs. Spectre is a broader class affecting all CPUs with speculative execution and requires per-variant mitigations.

In [ ]:
COMPARISON = {
    'Attack':          ('Meltdown',                        'Spectre'),
    'CVE':             ('CVE-2017-5754',                   'CVE-2017-5753, CVE-2017-5715'),
    'Mechanism':       ('Out-of-order exec + page faults', 'Branch predictor mis-training'),
    'Crosses':         ('User -> Kernel boundary',         'Any trust boundary'),
    'Exploits':        ('CPU bug (fixed in hw)',            'CPU design feature (hard to fix)'),
    'Affected CPUs':   ('Intel (mainly), some ARM',        'All speculative execution CPUs'),
    'OS mitigation':   ('KPTI (kernel page table isol)',   'Retpoline, IBRS, IBPB'),
    'HW mitigation':   ('Fixed in post-2019 Intel CPUs',  'eIBRS, BHI mitigation in new CPUs'),
    'Perf impact':     ('5-30% on syscall-heavy workloads','Varies; retpoline ~3-5%'),
    'Browser fix':     ('Site isolation',                  'Reduced timer resolution + COOP/COEP'),
}

print(f'{"Property":<20} {"Meltdown":<40} {"Spectre"}')
print('-' * 90)
for prop, (mel, spe) in COMPARISON.items():
    print(f'{prop:<20} {mel:<40} {spe}')

In [ ]:
# KPTI mitigation simulation
class KPTISimulation:
    """Demonstrates how KPTI separates user/kernel page tables."""

    def __init__(self):
        # Without KPTI: both kernel and user mappings in same page table
        self.unified_pagetable = {
            'user_stack':    (0x7fff1000, 'user', 'rw'),
            'user_code':     (0x400000,   'user', 'rx'),
            'kernel_text':   (0xffffffff80000000, 'kernel', 'rx'),
            'kernel_data':   (0xffffffff81000000, 'kernel', 'rw'),
            'kernel_secrets':(0xffffffff82000000, 'kernel', 'r'),
        }

        # With KPTI: user page table has minimal kernel mappings
        self.user_pagetable   = {
            'user_stack':    (0x7fff1000, 'user', 'rw'),
            'user_code':     (0x400000,   'user', 'rx'),
            'trampoline':    (0xfffffe0000000000, 'kernel', 'rx'),  # only syscall trampoline
        }
        self.kernel_pagetable = self.unified_pagetable  # full access

    def meltdown_attempt(self, target_name: str, use_kpti: bool) -> str:
        table = self.user_pagetable if use_kpti else self.unified_pagetable
        if target_name in table:
            addr, priv, perm = table[target_name]
            if priv == 'kernel':
                if use_kpti:
                    return f'BLOCKED: {target_name} not in user page table (KPTI)'
                else:
                    return f'EXPOSED: {target_name} @ {hex(addr)} in shared table!'
            return f'OK: user-accessible {target_name}'
        return f'NOT MAPPED: {target_name}'

kpti = KPTISimulation()
targets = ['user_stack', 'kernel_text', 'kernel_secrets']

print('Meltdown attack surface:')
print(f'{"Target":<20} {"Without KPTI":<40} {"With KPTI"}')
print('-' * 90)
for t in targets:
    no_kpti  = kpti.meltdown_attempt(t, use_kpti=False)
    with_kpti = kpti.meltdown_attempt(t, use_kpti=True)
    print(f'{t:<20} {no_kpti:<40} {with_kpti}')